In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. the full gpt-oss-20b vocabulary
#
# This used to read list_all_tokens.npy: 28,973 strings, a genuine but partial
# slice of the model at 14.5% of it. That subset was missing real direction
# tokens (' arriba', ' oben', ' unten', ' kiri') and, worse, was missing '上'
# while keeping 下/左/右 -- so UP was asymmetrically under-covered in CJK purely
# as an artifact of the filtering. Start from every token the model has.
import os
from transformers import AutoTokenizer

MODEL_ID = "openai/gpt-oss-20b"
VOCAB_NPY = "/media/alex/D/Uni/northeastern/data/jlens/vocab_all_tokens.npy"

if os.path.exists(VOCAB_NPY):
    list_all_tokens = np.load(VOCAB_NPY, allow_pickle=True)
else:
    _vtok = AutoTokenizer.from_pretrained(MODEL_ID)
    list_all_tokens = np.array([_vtok.decode([i]) for i in range(len(_vtok))],
                               dtype=object)
    np.save(VOCAB_NPY, list_all_tokens)

# kept only to report what widening the net actually bought
OLD_SUBSET = set(
    str(t) for t in
    np.load("/media/alex/D/Uni/northeastern/data/jlens/list_all_tokens.npy",
            allow_pickle=True)
)

print(f"{len(list_all_tokens)} tokens, {len(set(map(str, list_all_tokens)))} unique strings")
print(f"previous subset: {len(OLD_SUBSET)} "
      f"({100 * len(OLD_SUBSET) / len(list_all_tokens):.1f}% of the vocabulary)")

200019 tokens, 198816 unique strings
previous subset: 28973 (14.5% of the vocabulary)


In [3]:
import re
from difflib import SequenceMatcher

# 2. seeds
#
# LEX_SEEDS feeds difflib, so it is deliberately conservative: every short or
# English-homographic form is left out. Including them cost 7k false positives
# ('le', 'alt', 'sol', 'sus', 'len', 'opp', 'bal', 'der', 'port' all fire on
# ordinary code tokens; 'linke'/'linker'/'turun'/'onder'/'runter' pull in
# linked, line, turn, wonder, router at ratio .83-.91).
LEX_SEEDS = {
    "up": ["up", "upward", "upwards", "arriba", "subir", "dessus", "monter",
           "oben", "aufwaerts", "aufwärts", "sopra", "acima", "boven", "omhoog",
           "nahoru", "yukari", "yukarı", "epano", "πάνω", "вверх", "вгору", "ऊपर"],
    "down": ["down", "downward", "downwards", "abajo", "abaixo", "bajar",
             "dessous", "descendre", "unten", "herunter", "abwaerts", "abwärts",
             "beneden", "omlaag", "asagi", "aşağı", "κάτω", "вниз", "नीचे"],
    "left": ["left", "leftward", "leftwards", "izquierda", "izquierdo", "gauche",
             "sinistra", "sinistro", "esquerda", "esquerdo", "vanster", "vänster",
             "vasen", "doleva", "vlevo", "stanga", "stânga", "balra", "aristera",
             "αριστερά", "влево", "слева", "ліворуч", "बाएं"],
    "right": ["right", "rightward", "rightwards", "derecha", "derecho", "droite",
              "rechts", "rechte", "destra", "destro", "direita", "direito",
              "hoger", "höger", "oikea", "prawo", "prawa", "vpravo", "doprava",
              "dreapta", "jobbra", "deksia", "δεξιά", "вправо", "справа", "दाएं"],
}

# SEM_ANCHORS feeds the embedder, which scores meaning rather than characters,
# so ambiguous *spellings* are safe here. Two things are not.
#
# AMBIGUOUS MEANINGS. Auditing which anchor admitted which token caught these:
#   góra  -> Polish "up" AND "mountain": berg, hillside, Everest (12/12 junk)
#   prawo -> Polish "right" AND "law":   law, LAW, ley, legal (19)
#   links -> German "left" AND English "hyperlink": link, /link, (link
#   sol   -> Turkish "left" AND sun / musical note
#   bas   -> French "down" AND English bass: Bass, bassist, bask, bast
# Bare words are therefore replaced by phrases that pin the directional sense
# ("en bas", "in giù", "al di sopra"), which costs a little recall on the bare
# form but removes the whole contaminated cluster.
#
# HUB ANCHORS. Bare single syllables sit close to everything: '위' averaged .713
# cosine against the whole vocabulary, and with '아래' admitted 4,843 of 5,207
# semantic hits, mostly 1-3 char subword fragments. The two-syllable 위쪽 /
# 아래쪽 ("upper side" / "lower side") are far more specific.
#
# Arrows go to all four directions: with only ← and → present, '↑' had no
# correct home and was landing under RIGHT.
SEM_ANCHORS = {
    "up":    ["up", "upward", "above", "arriba", "en haut", "oben", "al di sopra",
              "em cima", "boven", "uppåt", "w górę", "вверх", "yukarı", "上",
              "위쪽", "di atas", "↑"],
    "down":  ["down", "downward", "below", "abajo", "en bas", "unten", "in giù",
              "em baixo", "beneden", "nedåt", "w dół", "вниз", "aşağı", "下",
              "아래쪽", "di bawah", "↓"],
    "left":  ["left", "izquierda", "gauche", "nach links", "a sinistra",
              "à esquerda", "vänster", "w lewo", "влево", "sol taraf", "左",
              "왼쪽", "ke kiri", "←"],
    "right": ["right", "derecha", "droite", "rechts", "a destra", "à direita",
              "höger", "w prawo", "вправо", "sağ taraf", "右", "오른쪽",
              "ke kanan", "→"],
}

print({d: len(v) for d, v in LEX_SEEDS.items()},
      {d: len(v) for d, v in SEM_ANCHORS.items()})

{'up': 22, 'down': 19, 'left': 24, 'right': 26} {'up': 17, 'down': 17, 'left': 14, 'right': 14}


In [4]:
# 3. token -> word pieces
SEP_RE = re.compile(r"""[_\-./\\:,;()\[\]{}<>"'`|!?*+=#@$%^&~\s]+""")
CAMEL_RE = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def strip_token(tok):
    """Surface form with whitespace/punctuation shaved off: '_LEFT' -> 'LEFT'."""
    return SEP_RE.sub(" ", tok).strip()


def parts_of(tok):
    """Word pieces to match on, lowercased: '.moveLeft' -> ['move', 'left', 'moveleft']."""
    out = []
    for piece in SEP_RE.split(tok.strip()):
        if piece:
            out.extend(p for p in CAMEL_RE.split(piece) if p)
    whole = SEP_RE.sub("", tok.strip())
    if whole:
        out.append(whole)
    return list(dict.fromkeys(p.lower() for p in out))


parts_of("_LEFT"), parts_of(" moveRight"), parts_of("/right")

(['left'], ['move', 'right', 'moveright'], ['right'])

In [5]:
# 4. lexical stage: difflib over the word pieces
#
# Two guards:
#
# (a) the ratio cutoff scales with seed length. .80 between 5-char strings just
#     means "one char differs" (light/right, line/linke, turn/turun), so short
#     seeds demand exact equality and everything else .85.
#
# (b) an English part must match essentially exactly. Raw ratio cannot separate
#     ischierda/izquierda (.667) from counter/herunter (.667) -- identical
#     scores, and no length or prefix guard splits them. What splits them is
#     that the junk side is a real English word needing no direction to explain
#     it, while a mangled foreign direction token is not in the dictionary.
#
# There used to be a permissive .62 tier for long-vs-long pairs, to catch
# mangled foreign spellings. It does not survive the full vocabulary: at 29k
# tokens it cost 23 false positives, at 200k it cost 726, with four seeds
# generating two thirds of them (aristera 186, sinistra 110, descendre 99,
# herunter 99) and LEFT ending up 491 lexical-only hits out of 531.
ENGLISH = {w.strip().lower()
           for w in open("/usr/share/dict/american-english", encoding="latin-1")
           if w.strip()}

flat_lex = [(w, d) for d, ws in LEX_SEEDS.items() for w in ws]


def min_ratio(part, seed):
    if part in ENGLISH:
        return 0.95               # an English word is already explained; demand near-identity
    if len(seed) <= 4:
        return 1.0                # exact only: 'up' scores .80 against 'cup'
    return 0.85


# 200k tokens x 91 seeds is ~36M SequenceMatcher calls, so two cheap outs:
# ratio() cannot exceed 2*min(la,lb)/(la+lb), which rules out most pairs on
# length alone, and parts repeat heavily across the vocabulary so results are
# memoised per part rather than per token.
_part_cache = {}


def score_part(part):
    hit = _part_cache.get(part)
    if hit is not None:
        return hit
    best, bw, bd = 0.0, "", ""
    lp = len(part)
    for w, d in flat_lex:
        floor = min_ratio(part, w)
        if 2 * min(lp, len(w)) / (lp + len(w)) < floor:
            continue              # cannot reach the cutoff whatever the overlap
        r = SequenceMatcher(None, part, w).ratio()
        if r >= floor and r > best:
            best, bw, bd = r, w, d
    _part_cache[part] = (best, bw, bd)
    return _part_cache[part]


lex_score, lex_seed, lex_dir = [], [], []
for n, t in enumerate(list_all_tokens):
    best, bw, bd = 0.0, "", ""
    for part in parts_of(str(t)):
        r, w, d = score_part(part)
        if r > best:
            best, bw, bd = r, w, d
    lex_score.append(best); lex_seed.append(bw); lex_dir.append(bd)
    if n % 50000 == 0:
        print(f"  {n:>6d}/{len(list_all_tokens)}  ({len(_part_cache)} parts cached)")
lex_score = np.array(lex_score)

print(f"\n{(lex_score > 0).sum()} tokens matched lexically")
for lo, hi in [(0.99, 1.01), (0.90, 0.99), (0.85, 0.90)]:
    idx = np.where((lex_score >= lo) & (lex_score < hi))[0]
    print(f"\n=== [{lo}, {hi}) : {len(idx)} ===")
    print("   ", [repr(str(list_all_tokens[i])) for i in idx[:40]])

       0/200019  (0 parts cached)


   50000/200019  (31559 parts cached)


  100000/200019  (63766 parts cached)


  150000/200019  (96628 parts cached)


  200000/200019  (129702 parts cached)

219 tokens matched lexically

=== [0.99, 1.01) : 121 ===
    ["'up'", "' up'", "' right'", "' down'", "'right'", "'Up'", "' left'", "'left'", "'down'", "' Up'", "'UP'", "'Down'", "'-up'", "'Left'", "'Right'", "' Down'", "'-right'", "'.left'", "' Right'", "'-left'", "'.right'", "'_up'", "' UP'", "'_left'", "'_right'", "'RIGHT'", "'-down'", "' Left'", "'_UP'", "' boven'", "'(left'", "'.up'", "' direito'", "' derecho'", "'_down'", "'.Right'", "'.Left'", "'_LEFT'", "'_DOWN'", "' abaixo'"]

=== [0.9, 0.99) : 43 ===
    ["' recht'", "' destr'", "' droit'", "' desta'", "'recht'", "'estro'", "' справ'", "' derechos'", "' echte'", "' montr'", "' права'", "' Recht'", "' право'", "' direitos'", "' baixo'", "'onter'", "' punten'", "'punten'", "' montrer'", "' rechten'", "' rechter'", "' Montr'", "'untegn'", "' direto'", "'baixo'", "' desto'", "' hogere'", "' Derechos'", "'estra'", "' direta'", "' pravo'", "' arribar'", "'auche'", "'rechten'", "'rechter'", "'

In [6]:
# 5. embedding model (CPU: the GTX 1050 is sm_61, unsupported by this torch build)
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
_tok = AutoTokenizer.from_pretrained(MODEL)
_mod = AutoModel.from_pretrained(MODEL).eval()


@torch.no_grad()
def encode(texts, batch=256, show_every=None):
    """Mean-pooled, L2-normalised embeddings. Bare strings, no template --
    'the direction "{x}"' compresses everything upward and kills the separation."""
    out = []
    for i in range(0, len(texts), batch):
        b = _tok(texts[i:i + batch], padding=True, truncation=True,
                 max_length=16, return_tensors="pt")
        h = _mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        out.append(F.normalize((h * m).sum(1) / m.sum(1), dim=-1))
        if show_every and (i // batch) % show_every == 0:
            print(f"  {i + len(b['input_ids']):>6d}/{len(texts)}")
    return torch.cat(out).numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
# 6. semantic stage: centred embeddings, cosine against the anchors, rank-normalised
#
# Hubness bites on both sides of the comparison and each side needs its own fix.
#
# TOKEN SIDE -- the embedding space is anisotropic: every vector carries a large
# shared component, so short subword fragments come out similar to everything.
# Raw cosine >= .85 kept 38,679 of 150,233 forms, which is why no cosine
# threshold could ever work. Subtracting the mean form embedding removes that
# common direction and the top matches become almost purely genuine direction
# words in every script.
#
# ANCHOR SIDE -- anchors differ wildly in how close they sit to the vocabulary
# ('위' averaged .713, 'down' .357), so a max over anchors is a vote for the
# hubbiest one. Two normalisations were tried and each fixed only half:
#   z = (cos-mean)/std     damps hubs but leaves each anchor a different
#     ceiling. A perfect cos=1.0 scores +6.16 on 'down', +2.79 on 'right',
#     +2.08 on 'sağ' -- no RIGHT token could clear a global z=3.0 however
#     perfect, while junk like 'levantar' scored +3.99.
#   (cos-mean)/(1-mean)    equalises the ceiling at 1.0 but drops the damping,
#     and dividing by a small (1-mean) amplifies instead: '위' alone then
#     admitted 4,261 tokens, mostly 1-3 char fragments.
# Rank has both properties and assumes no distribution: a perfect match is the
# top percentile for every anchor whatever its hubbiness, and only the top k per
# anchor can be admitted.
#
# "Which direction is it?" -> raw (centred) cosine, unnormalised. Normalising
# here over-corrects: '↑' matched anchor '↑' at cos=1.000 but only z=+2.88,
# versus '→' at cos=.725 yet z=+3.61, so a perfect match lost and '↑' was filed
# under RIGHT.
EMB_NPY = "/media/alex/D/Uni/northeastern/data/jlens/vocab_form_embeddings.npy"

anchor_words = [w for ws in SEM_ANCHORS.values() for w in ws]
anchor_dir = [d for d, ws in SEM_ANCHORS.items() for _ in ws]

# many tokens collapse to the same stripped form ('left', ' left', '_left', '.left')
uniq = sorted({strip_token(str(t)) for t in list_all_tokens} - {""})

# 150k forms is ~40 min on CPU, so the matrix is cached; delete the .npy to redo
if os.path.exists(EMB_NPY):
    U = np.load(EMB_NPY)
    assert len(U) == len(uniq), "cache is stale, delete it"
    print(f"loaded cached embeddings {U.shape}")
else:
    U = encode(uniq, batch=256, show_every=100)
    np.save(EMB_NPY, U)

A = encode(anchor_words)


def unit(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


centre = U.mean(0, keepdims=True)                       # the common component
S = unit(U - centre) @ unit(A - centre).T               # (n_uniq, n_anchors)
RANK = S.argsort(0).argsort(0) / (len(uniq) - 1)        # percentile per anchor

i_r = RANK.argmax(axis=1)                               # how strong -> threshold
i_c = S.argmax(axis=1)                                  # which way  -> label
by_form = {f: (float(RANK[i, i_r[i]]), anchor_words[i_c[i]], anchor_dir[i_c[i]])
           for i, f in enumerate(uniq)}

sem_rank, sem_seed, sem_dir = [], [], []
for t in list_all_tokens:
    r, w, d = by_form.get(strip_token(str(t)), (0.0, "", ""))
    sem_rank.append(r); sem_seed.append(w); sem_dir.append(d)
sem_rank = np.array(sem_rank)

print("\nsanity -- the literal direction words and the arrows:")
for a in ["up", "down", "left", "right", "↑", "↓", "←", "→"]:
    if a in by_form:
        s, w, d = by_form[a]
        print(f"  {a:6s} -> {d.upper():5s} (via {w}, rank={s:.5f})")
print(f"\nembedded {len(uniq)} unique forms for {len(list_all_tokens)} tokens")

loaded cached embeddings (150233, 384)



sanity -- the literal direction words and the arrows:
  up     -> UP    (via up, rank=1.00000)
  down   -> DOWN  (via down, rank=1.00000)
  left   -> LEFT  (via left, rank=1.00000)
  right  -> RIGHT (via right, rank=1.00000)
  ↑      -> UP    (via ↑, rank=1.00000)
  ↓      -> DOWN  (via ↓, rank=1.00000)
  ←      -> LEFT  (via ←, rank=1.00000)
  →      -> RIGHT (via →, rank=1.00000)

embedded 150233 unique forms for 200019 tokens


In [8]:
# 7. combine both signals into one frame
LEX_T = 0.01     # the guards in cell 4 already zeroed everything below bar
SEM_T = 0.9999   # percentile: top .01% of the vocabulary for some anchor.
                 # Inspecting the ranked list, forms 45-220 are near-uniformly
                 # genuine, 220-306 degrade, and past ~306 it is mostly code
                 # vocabulary (Font, Bash, DESCRIPTOR, Operator, syntax, Pdf).

df = pd.DataFrame({
    "token": [str(t) for t in list_all_tokens],
    "lex_score": lex_score, "lex_seed": lex_seed, "lex_dir": lex_dir,
    "sem_rank": sem_rank, "sem_seed": sem_seed, "sem_dir": sem_dir,
})
# cached so SEM_T can be retuned without recomputing anything
df.to_pickle("/media/alex/D/Uni/northeastern/data/jlens/direction_scores.pkl")

print("semantic bands (where to put SEM_T):")
for lo, hi in [(0.99995, 1.01), (0.9999, 0.99995), (0.9995, 0.9999),
               (0.999, 0.9995), (0.998, 0.999)]:
    sel = df[(df.sem_rank >= lo) & (df.sem_rank < hi)]
    print(f"  [{lo:.5f}, {hi:.5f}) : {len(sel):5d}  {[repr(t) for t in sel.token.head(12)]}")

print(f"\nlexical  : {(df.lex_score >= LEX_T).sum():5d} tokens")
print(f"semantic : {(df.sem_rank >= SEM_T).sum():5d} tokens")
print(f"union    : {((df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)).sum():5d} tokens")

semantic bands (where to put SEM_T):
  [0.99995, 1.01000) :   288  ["'up'", "' up'", "' right'", "' down'", "'right'", "'Up'", "' left'", "'下'", "'上'", "'left'", "'down'", "' Up'"]
  [0.99990, 0.99995) :   143  ["'Com'", "' Com'", "'ин'", "' ин'", "'＿'", "' operator'", "' beyond'", "'.Com'", "'igu'", "'йн'", "'iendo'", "'operator'"]
  [0.99950, 0.99990) :  1117  ["'\\x00'", "'्'", "'्�'", "'ner'", "'્'", "'્�'", "' high'", "' here'", "'ga'", "'ث'", "'input'", "' writ'"]
  [0.99900, 0.99950) :  1444  ["'в'", "'turn'", "' в'", "'ri'", "'್'", "'Type'", "'್�'", "'ze'", "'ụ'", "'\\xad'", "'Override'", "'Code'"]
  [0.99800, 0.99900) :  3074  ["'op'", "'ی'", "'ı'", "'ь'", "'ide'", "'ð'", "'put'", "'ft'", "'String'", "'text'", "'ν'", "'，'"]

lexical  :   219 tokens
semantic :   431 tokens
union    :   539 tokens


In [9]:
# 8. final candidate set
review = df[(df.lex_score >= LEX_T) | (df.sem_rank >= SEM_T)].copy()
review["hit"] = np.where(
    (review.lex_score >= LEX_T) & (review.sem_rank >= SEM_T), "both",
    np.where(review.lex_score >= LEX_T, "lexical", "semantic"),
)
review["direction"] = np.where(review.lex_score >= LEX_T, review.lex_dir, review.sem_dir)
review["rank"] = review[["lex_score", "sem_rank"]].max(axis=1)
review = review.sort_values(["direction", "hit", "rank"], ascending=[True, True, False])

print(f"{len(review)} candidates  "
      f"(both={sum(review.hit == 'both')}, "
      f"lexical only={sum(review.hit == 'lexical')}, "
      f"semantic only={sum(review.hit == 'semantic')})")
print(pd.crosstab(review.direction, review.hit).to_string())

# 'both' is essentially all true positives; the single-signal groups are where
# to spend review effort.
pd.set_option("display.max_rows", 800)
review[["token", "direction", "hit", "lex_score", "lex_seed", "sem_rank", "sem_seed"]]

539 candidates  (both=111, lexical only=108, semantic only=320)
hit        both  lexical  semantic
direction                         
down         28       20       107
left         21       16        27
right        31       49        68
up           31       23       118


,token,direction,hit,lex_score,lex_seed,sem_rank,sem_seed
1917,down,down,both,1.000000,down,1.000000,down
4653,down,down,both,1.000000,down,1.000000,down
6064,Down,down,both,1.000000,down,0.999993,down
8755,Down,down,both,1.000000,down,0.999993,down
26673,-down,down,both,1.000000,down,1.000000,down
40267,_down,down,both,1.000000,down,1.000000,down
46368,_DOWN,down,both,1.000000,down,0.999987,down
46979,abaixo,down,both,1.000000,abaixo,0.999967,abajo
49412,DOWN,down,both,1.000000,down,0.999987,down
54122,.down,down,both,1.000000,down,1.000000,down


In [10]:
# 9. final output: {UP, DOWN, LEFT, RIGHT} -> raw, unprocessed tokens
#
# Everything above (stripping, lowercasing, camel splitting) was scoring
# machinery only. What lands here is the byte-exact vocabulary string --
# leading spaces, tabs, punctuation and all.
import json

direction_tokens = {
    d.upper(): review.loc[review.direction == d, "token"].drop_duplicates().tolist()
    for d in ["up", "down", "left", "right"]
}

originals = set(str(t) for t in list_all_tokens)
flat = [t for v in direction_tokens.values() for t in v]
for k, v in direction_tokens.items():
    assert all(t in originals for t in v), f"{k}: token was modified"
assert len(set(flat)) == len(flat), "token assigned to two directions"

print(f"{'':6s} {'n':>5s} {'new':>5s}   sample")
for k, v in direction_tokens.items():
    new = [t for t in v if t not in OLD_SUBSET]
    print(f"{k:6s} {len(v):5d} {len(new):5d}   {[repr(t) for t in v[:6]]}")
    if new:
        print(f"{'':13s} newly reachable: {[repr(t) for t in new[:12]]}")

OUT = "/media/alex/D/Uni/northeastern/data/jlens/direction_tokens_full.json"
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(direction_tokens, f, ensure_ascii=False, indent=2)

# round-trip check: the file must give back byte-identical strings
with open(OUT, encoding="utf-8") as f:
    assert json.load(f) == direction_tokens
print(f"\n{len(flat)} tokens -> {OUT}")

           n   new   sample
UP       172   111   ["'up'", "' up'", "'Up'", "' Up'", "'UP'", "'-up'"]
              newly reachable: ["' boven'", "' cima'", "' oben'", "'oben'", "' arriba'", "' ऊपर'", "'boven'", "' omhoog'", "' πάνω'", "' вверх'", "' subir'", "' dessus'"]
DOWN     155   108   ["' down'", "'down'", "'Down'", "' Down'", "'-down'", "'_down'"]
              newly reachable: ["' unten'", "'-dessous'", "' bajar'", "' नीचे'", "' aşağı'", "' dessous'", "'unten'", "' beneden'", "' κάτω'", "' baixo'", "'baixo'", "' herunter'"]
LEFT      64    33   ["' left'", "'left'", "'Left'", "'.left'", "'-left'", "'_left'"]
              newly reachable: ["' izquierdo'", "' esquer'", "' oleva'", "'auche'", "'bara'", "' bara'", "'asen'", "' bala'", "' Bara'", "' suministro'", "' Bala'", "' ministro'"]
RIGHT    148    98   ["' right'", "'right'", "'Right'", "'-right'", "' Right'", "'.right'"]
              newly reachable: ["' direito'", "' Rechts'", "'rechte'", "'rechts'", "' rechte'", "' спра